In [13]:
import pandas as pd
import numpy as np
import re
import json
from functools import reduce

with open('modelos.json') as f:
    modelos = json.load(f)

In [14]:
def ajusta_respostas(df_dados):
    df_ajuste = pd.read_excel('../_dados/Memória de ajuste do Questionário.xlsx')
    for idx, ajuste in df_ajuste.iterrows():
        df_dados.at[ajuste['orgao'], ajuste['item']] = ajuste['resposta_ajustada']
    
    df_dados = df_dados.replace('Não se aplica.', 'Adota em maior parte ou totalmente.')
    
    return df_dados

def trata_respostas(df_dados):
    df_dados = df_dados.filter(regex=r'\d{4}')
    df_dados = df_dados.filter(regex=r'^((?!lei|est|evi|nsa|raz|SQ).)*$')
    df_dados.columns = [re.sub(r"\.|\[|\]|ext", '', col) for col in df_dados.columns]
    
    df_dados = df_dados.replace(['N/A', 'Nao', 'Não'], '0')
    df_dados = df_dados.fillna(0)
    df_dados = df_dados.replace('Sim', '1')
    
    df_dados = ajusta_respostas(df_dados)
    
    df_dados = df_dados.replace('Adota em maior parte ou totalmente.', 'Adota em maior parte ou totalmente')
    df_dados = df_dados.replace('Adota parcialmente.', 'Adota parcialmente')    
    df_dados = df_dados.replace('Adota em menor parte.', 'Adota em menor parte')
    df_dados = df_dados.replace('Há decisão formal ou plano aprovado para adotá-lo.', 'Há decisão formal ou plano aprovado para adotá-lo')
    df_dados = df_dados.replace('Não adota.', 'Não adota')
    df_dados = df_dados.replace('Não se aplica.', 'Não se aplica')
    
    # Convert columns with uppercase letters to numeric
    df_dados[df_dados.columns[df_dados.columns.str.isupper()]] = df_dados[df_dados.columns[df_dados.columns.str.isupper()]].apply(pd.to_numeric, errors='coerce')

    return df_dados

def unique(li):
    return reduce(lambda re, x: re+[x] if x not in re else re, li, [])

#Converte lista de listas para lista
def flatten(A):
    rt = []
    for i in A:
        if isinstance(i,list): rt.extend(flatten(i))
        else: rt.append(i)
    return rt

def get_id_questoes_igovti(modelos):
    id_questoes_igovti = []
    for agregado in modelos.keys():
        id_questoes_igovti.append(modelos[agregado]['variaveis'])

    id_questoes_igovti = flatten(id_questoes_igovti)
    #Remove os valores nao numericos
    id_questoes_igovti = [re.sub(r'\D', '', i) for i in id_questoes_igovti]
    #Remove os itens unicos que nao sejam numeros
    return unique([a for a in id_questoes_igovti if a.isnumeric() ])

def calcula_questao(row):
    niveis = {
        "Não adota": 0.00,
        "Há decisão formal ou plano aprovado para adotá-lo": 0.05,
        "Adota em menor parte": 0.15,
        "Adota parcialmente": 0.50,
        "Não se aplica": 0.50,
        "Adota": 1.00,
        "Adota em maior parte ou totalmente": 1.00
    }
    
    resposta_base = row[0].replace('.','')
    questoes_adicionais = pd.to_numeric(row[1:])
    
    if resposta_base == 'Adota parcialmente' or resposta_base == 'Adota em maior parte ou totalmente':
        pbase = niveis[resposta_base]
        desconto_max = 0.85 if pbase == 1 else 0.35
        desconto = ((1 - questoes_adicionais).sum() * desconto_max) / len(questoes_adicionais)
        
        row[0] = pbase - desconto
        
    else:
        row[0] = niveis[resposta_base]
        
    return row

In [17]:
df_setic_bruto = pd.read_excel('dados/resposta_questionario_setic.xlsx').set_index('attribute_1')
df_setic = trata_respostas(df_setic_bruto)

id_questoes = get_id_questoes_igovti(modelos)

df_processado = pd.DataFrame()
for id_questao in id_questoes:
    df_questoes = df_setic.filter(regex=f'{id_questao}', axis=1)
    df_questoes_descontado = df_questoes.apply(calcula_questao, axis=1)
    df_processado = pd.concat([df_processado, df_questoes_descontado], axis=1)

for agregado_nome in modelos.keys():
    #print(agregado_nome, modelos[agregado_nome]['variaveis'])
    #Pega os componentes que formam o agregado e adiciona um 'q' no inicio de cada numero
    agregado_componentes = [v if v.isalpha() else 'q' + v for v in modelos[agregado_nome]['variaveis']]
    
    df_agregado = df_processado[agregado_componentes].values.astype(float)    
    df_processado[agregado_nome] = np.dot(df_agregado, modelos[agregado_nome]['peso'])
    
#df_processado = pd.concat([df_setic_bruto.attribute_1, df_processado], axis=1)
df_processado.to_excel('dados/resultado_igovti.xlsx')

In [16]:
df_processado

,q1133,q1133A,q1133B,q1133C,q1133D,q1133E,q1133F,q1131,q1131A,q1131B,...,ProcessoSegInfo,ProcessoSoftware,iGestProjetosTI,iGestContratosTI,GovernancaTI,iGestSegInfo,ProcessosTI,iGestTI,iGovTI,ProcessosContratacao
attribute_1,,,,,,,,,,,,,,,,,,,,,
FAPERJ,0.150000,0,0,0,0,0,0,0.000000,0,0,...,0.057644,0.00,0.00,0.706748,0.032416,0.093913,0.103873,0.309156,0.170786,0.292097
SEAPPA,0.208333,0,0,1,0,0,0,0.208333,0,0,...,0.048508,0.00,0.00,0.425269,0.182889,0.010658,0.062147,0.064342,0.123615,0.449834
TURISRIO,0.208333,1,0,0,0,0,0,0.325000,0,1,...,0.161837,0.00,0.15,0.083786,0.151265,0.105250,0.145238,0.212981,0.182123,0.418904
ISP,0.575000,0,0,1,1,1,0,0.716667,1,1,...,0.735192,0.00,0.15,0.887550,0.303682,0.364219,0.267827,0.654021,0.478851,0.727754
SEFAZ,0.150000,0,0,0,0,0,0,0.150000,0,0,...,0.340985,1.00,1.00,0.925033,0.165158,0.315391,0.518833,0.521578,0.343368,0.666519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
IASERJ,0.500000,0,0,0,0,0,0,0.500000,0,0,...,0.150000,0.15,0.15,0.046303,0.249267,0.111933,0.132801,0.160833,0.205050,0.150000
UERJ,0.150000,0,0,0,0,0,0,0.150000,0,0,...,0.023016,0.15,0.00,0.046303,0.098347,0.023710,0.054688,0.092672,0.095509,0.150000
EMATER,0.325000,1,0,1,0,1,0,0.500000,0,0,...,0.180357,0.05,0.49,0.046303,0.243926,0.153254,0.180041,0.161065,0.202496,0.133283


## Memória de Cálculo

In [22]:
for agregado_nome in modelos.keys():
    print(agregado_nome)
    componentes = []
    pesos = modelos[agregado_nome]['peso']
    pesos = pesos if isinstance(pesos, list) else [pesos]
    
    for i, item in enumerate(modelos[agregado_nome]['variaveis']):
        componentes.append(f"{item} * {'{:.4f}'.format(pesos[i])} ")
    print('+ '.join(componentes))

ResultadoTI
1133 * 0.3274 + 1131 * 0.3099 + 1132 * 0.3627 
ModeloTI
1111 * 1.0000 
MonitorAvaliaTI
1121 * 0.2400 + 1122D * 0.3804 + 1122E * 0.3796 
PessoasTI
2211 * 0.2000 + 2212 * 0.1897 + 2213 * 0.1386 + 2221 * 0.1798 + 2231 * 0.1675 + 2241 * 0.1243 
PlanejamentoTI
2111 * 0.5000 + 2112 * 0.5000 
iGestServicosTI
2121 * 0.2363 + 2122 * 0.2630 + 2123 * 0.2356 + 2124 * 0.2651 
iGestNiveisServicoTI
2131 * 1.0000 
iGestRiscosTI
2141 * 0.1352 + 2142 * 0.1144 + 2146 * 0.1607 + 2147 * 0.1546 + 2143 * 0.1729 + 2144 * 0.1654 + 2145 * 0.0969 
EstruturaSegInfo
2151 * 0.3372 + 2152 * 0.3286 + 2153 * 0.3342 
ProcessoSegInfo
2161 * 0.1691 + 2162 * 0.1819 + 2163 * 0.1684 + 2164 * 0.1468 + 2165 * 0.1803 + 2166 * 0.1534 
ProcessoSoftware
2171 * 1.0000 
iGestProjetosTI
2181 * 1.0000 
iGestContratosTI
2311 * 0.3087 + 2321A * 0.3587 + 2322A * 0.3327 
GovernancaTI
ModeloTI * 0.3425 + MonitorAvaliaTI * 0.3444 + ResultadoTI * 0.3132 
iGestSegInfo
EstruturaSegInfo * 0.1852 + ProcessoSegInfo * 0.2197 + 2145 * 

In [4]:
questoes_igovti = 'q'+'$|q'.join(id_questoes) + '$'
teste = df_setic.filter(regex=questoes_igovti)
#teste.apply(pd.Series.value_counts, axis=1).sum(axis=1)
pd.concat([df_processado['attribute_1'], df_setic['q1011'], teste.apply(pd.Series.value_counts, axis=1)/44], axis=1).sort_values(by='Não se aplica', ascending=False).head(10)

,attribute_1,q1011,Adota em maior parte ou totalmente,Adota em menor parte,Adota parcialmente,Há decisão formal ou plano aprovado para adotá-lo,Não adota,Não se aplica
14,CASERJ,0,NaN,NaN,NaN,NaN,NaN,1.000000
58,GVG,0,NaN,NaN,NaN,NaN,0.045455,0.954545
68,IASERJ,0,0.022727,NaN,NaN,0.090909,NaN,0.886364
41,SEIC,1,0.250000,0.090909,0.045455,NaN,0.022727,0.590909
27,SEAS,0,NaN,0.113636,0.068182,0.136364,0.113636,0.568182
8,SETD,1,0.363636,NaN,0.181818,0.136364,0.022727,0.295455
53,FIA/RJ,1,0.113636,NaN,NaN,0.113636,0.636364,0.136364
71,FLXIII,1,0.113636,0.136364,0.045455,0.090909,0.500000,0.113636
50,GSI,1,0.340909,0.113636,0.136364,NaN,0.318182,0.090909
3,ISP,1,0.636364,0.022727,0.022727,0.045455,0.181818,0.090909
